<a href="https://colab.research.google.com/github/vkhanht1/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkhanht1/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

- **Grain:** One row represents a unique `(query, page, date)` combination.
- **Time window:** Mid-panel month `2026-03` (March 2026) for training/verification.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Tables:** `search_console_url_impression`
- **Label:** `clicks > 0` in the next 7-day window.
- **Excluded:** Brand-specific query terms to avoid CTR bias.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
from huggingface_hub import list_repo_files, hf_hub_download
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
repo_id = "FlyRank/internship-warehouse"

all_files = list_repo_files(repo_id, repo_type="dataset", token=hf_token)
parquet_files = [f for f in all_files if f.endswith(".parquet")]

print(f"Total parquet files found: {len(parquet_files)}")
print("Sample files:", parquet_files[:5])

# Filter files for month 2026-03 (or grab the first parquet file found if no month tag exists in the name)
target_files = [f for f in parquet_files if "2026-03" in f]
if not target_files:
    target_files = parquet_files[:1] # Grab the first sample parquet file

print(f"\nDownloading files: {target_files}")

local_paths = []
for f in target_files:
    path = hf_hub_download(repo_id=repo_id, filename=f, repo_type="dataset", token=hf_token)
    local_paths.append(path)

# 2. Connect DuckDB
con = duckdb.connect()
con.sql(f"CREATE VIEW df_raw AS SELECT * FROM read_parquet({local_paths})")

# Print the list of actual columns in the table to verify column names
columns = [row[0] for row in con.sql("DESCRIBE df_raw").fetchall()]
print(f"\nList of columns available in data: {columns}\n")

# Automatically assign appropriate column names matching the actual data
query_col = "query" if "query" in columns else columns[0]
page_col = "page" if "page" in columns else ("url" if "url" in columns else columns[1])
date_col = "date" if "date" in columns else ("day" if "day" in columns else columns[2])

# Query 1: Grain Uniqueness Check
print("--- Query 1: Grain Uniqueness Check ---")
q1 = f"""
SELECT {query_col}, {page_col}, {date_col}, COUNT(*) as cnt
FROM df_raw
GROUP BY {query_col}, {page_col}, {date_col}
HAVING COUNT(*) > 1
LIMIT 5;
"""
print(con.sql(q1).df())

# Query 2: Row Count & Date Span
print("\n--- Query 2: Row Count & Date Span ---")
q2 = f"""
SELECT
    COUNT(*) as total_rows,
    MIN({date_col}) as min_date,
    MAX({date_col}) as max_date,
    COUNT(DISTINCT {query_col}) as unique_queries
FROM df_raw;
"""
print(con.sql(q2).df())

# Query 3: Availability Check with IS TRUE
print("\n--- Query 3: Availability Filter (IS TRUE) ---")
avail_col = "is_available" if "is_available" in columns else columns[0]
q3 = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(*) FILTER (WHERE {avail_col} IS NOT NULL) as available_rows
FROM df_raw;
"""
print(con.sql(q3).df())

Total parquet files found: 22
Sample files: ['dim_clients.parquet', 'dim_content.parquet', 'fact_content_daily_performance/month=2025-01/data_0.parquet', 'fact_content_daily_performance/month=2025-02/data_0.parquet', 'fact_content_daily_performance/month=2025-03/data_0.parquet']


List of columns available in data: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

--- Query 1: Grain Uniqueness Check ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, cnt]
Index: []

--- Query 2: Row Count & Date Span ---
   total_rows                  min_date                  max_date  \
0     9841378  content_000005d4ced12088  content_fffff09da8a25da6   

   unique_queries  
0              31  

--- Query 3: Availability Filter (IS TRUE) ---
   total_rows  available_rows
0     9841378         9841378


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **Limitation:** Reporting lag in GSC data sync (24-48h latency) and lack of intra-month external seasonality context.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

df = con.sql("SELECT * FROM df_raw LIMIT 1000").df()
columns = df.columns.tolist()

date_col = next((c for c in columns if any(k in c.lower() for k in ['date', 'day', 'dt', 'time'])), None)
page_col = next((c for c in columns if any(k in c.lower() for k in ['page', 'url', 'landing'])), None)
clicks_col = next((c for c in columns if any(k in c.lower() for k in ['click', 'session', 'visit'])), columns[-1])
impressions_col = next((c for c in columns if any(k in c.lower() for k in ['impression', 'view'])), clicks_col)
position_col = next((c for c in columns if any(k in c.lower() for k in ['position', 'rank'])), clicks_col)

# 3. Build 5 Features
# Feature 1: Historical CTR
df['hist_ctr_7d'] = pd.to_numeric(df[clicks_col], errors='coerce').fillna(0) / (pd.to_numeric(df[impressions_col], errors='coerce').fillna(0) + 1e-5)

# Feature 2: Average Position
df['avg_position_30d'] = pd.to_numeric(df[position_col], errors='coerce').fillna(0)

# Feature 3: Impression Volume
df['query_impression_volume_7d'] = pd.to_numeric(df[impressions_col], errors='coerce').fillna(0)

# Feature 4: URL Depth
if page_col:
    df['url_depth'] = df[page_col].astype(str).apply(lambda x: len(x.split('/')) - 1)
else:
    df['url_depth'] = 1

# Feature 5: Is Weekend
if date_col:
    dates_parsed = pd.to_datetime(df[date_col], errors='coerce')
    df['is_weekend'] = dates_parsed.dt.dayofweek.fillna(0) >= 5
else:
    df['is_weekend'] = False

# 4. Label & Features Matrix
df['label'] = (pd.to_numeric(df[clicks_col], errors='coerce').fillna(0) > 0).astype(int)
features = ['hist_ctr_7d', 'avg_position_30d', 'query_impression_volume_7d', 'url_depth', 'is_weekend']

# --- RUN EXPERIMENT ---
# 1. Honest Run
clf = RandomForestClassifier(random_state=42).fit(df[features], df['label'])
honest_auc = roc_auc_score(df['label'], clf.predict_proba(df[features])[:, 1])
print(f"Honest AUC Score: {honest_auc:.4f}")

# 2. Spring the Trap (Data Leakage)
df['leak'] = df['label'] + np.random.normal(0, 0.01, len(df))
clf_leak = RandomForestClassifier(random_state=42).fit(df[features + ['leak']], df['label'])
leaked_auc = roc_auc_score(df['label'], clf_leak.predict_proba(df[features + ['leak']])[:, 1])
print(f"Leaked AUC Score (The Trap): {leaked_auc:.4f}")

# 3. Cleanup Trap
df.drop(columns=['leak'], inplace=True)
print("Trap cleaned successfully!")

Honest AUC Score: 1.0000
Leaked AUC Score (The Trap): 1.0000
Trap cleaned successfully!


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.